In [1]:
import cv2
import numpy as np  
import open3d as o3d
import open3d.visualization as vis
import time
import threading 
import tkinter as tk

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
distance_threshold = 0.012      # трешхолд для поиска плоскости пола RANSAC
max_downsample_points = 200_000 # максимум точек, которые мы можем взять для поиск плоскости
ransac_iterations = 200_000     # количество итераций поиска пола
grid_step = 0.005               # размер клетки в метрах при разбиении пола на сетку
max_dist = 0.7                  # высота от пола, на которой точки считаются припятствованиями
diameter = 0.3                  # диаметр робота
morph_size = 5                  # количество пикселей для наращивания точек пола/препятствий
min_area = 5*diameter**2        # минимальная площадь отдельного участка, меньше которой нам не интересно
morph_fill_size = 11            # количество пикселей для устранения шума

In [9]:
#даунсемплинг через выбор рандомных точек
def downsample(pcd):
    if len(pcd.points) < max_downsample_points:
        return pcd
    random_points_count = max_downsample_points
    random_indices = np.random.choice(len(pcd.points), random_points_count, replace=False)
    return pcd.select_by_index(random_indices)

# Voxel-Даунсемплинг через numpy (без Open3D)
def voxel_downsample_fast(points, voxel_size):
    voxel_indices = np.floor(points / voxel_size).astype(np.int32)
    # Создаем уникальный ключ для каждого вокселя
    voxel_keys = (voxel_indices[:, 0] * 1000000 + 
                  voxel_indices[:, 1] * 1000 + 
                  voxel_indices[:, 2])
    _, unique_indices = np.unique(voxel_keys, return_index=True)
    return points[unique_indices]

#проектирование точек на плоскость
def project_to_plane(points, plane_model):
    [a, b, c, d] = plane_model
    normal = np.array([a, b, c])
    distances = np.dot(points, normal) + d

    projected_points = points - np.outer(distances, [a, b, c])
    return projected_points

#переход в систему координат плоскости (2 координаты)
def switch_to_plane_coords(projected, plane_v1, plane_v2):
    dot_v1 = np.dot(projected, plane_v1)
    dot_v2 = np.dot(projected, plane_v2)

    plane_coords = np.column_stack((dot_v1, dot_v2))
    x_max, x_min = np.max(plane_coords[:, 0]), np.min(plane_coords[:, 0])
    y_max, y_min = np.max(plane_coords[:, 1]), np.min(plane_coords[:, 1])
    return plane_coords, x_max, x_min, y_max, y_min

# неблокирующая визуализация np массива
def visualise(np_arr):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np_arr)
    show_pcd_concurrent(pcd)

# визуализация векторов
def visualise_vector_basis(pcd, normal, plane_v1, plane_v2):
    lines = []
    origin = (0,0,0)
    for vec, color in zip([normal, plane_v1, plane_v2], [[1,0,0], [0,1,0], [0,0,1]]):
        line = o3d.geometry.LineSet()
        line.points = o3d.utility.Vector3dVector([origin, origin + np.array(vec)])
        line.lines = o3d.utility.Vector2iVector([[0, 1]])
        line.colors = o3d.utility.Vector3dVector([color])
        lines.append(line)
    o3d.visualization.draw_geometries([pcd] + lines)

# функция для отсчета времени
def timer(info):
    global cur_time
    print(f"{info}: {-(cur_time - (cur_time := time.time())):.4f} секунд")

# просмотр pcd (вызывается в отдельном потоке функцией show_pcd_concurrent)
def show_pcd(pcd):
    window = vis.Visualizer()
    window.create_window(window_name="Open3d", width=1600, height=1200)
    window.add_geometry(pcd)
    while True:
        window.poll_events()
        window.update_renderer()
        time.sleep(0.001)
        try:
            if not window.poll_events():
                break
        except:
            break
    window.destroy_window()

# неблокирующий просмотр pcd
def show_pcd_concurrent(pcd):
    thread = threading.Thread(target=show_pcd, args=[pcd])
    thread.start()
    return thread

# просмотр изображений(np массивов)
def show_images(images_arr):
    root = tk.Tk()
    n = len(images_arr)
    w = root.winfo_screenwidth()
    root.destroy()
    gh, gw = images_arr[0].shape
    scale = (w/n) / gw
    new_w = int(gw * scale)
    new_h = int(gh * scale)

    cur_x = 0
    for i in range(n):
        win_name = 'Win' + str(i)
        cv2.namedWindow(win_name, cv2.WINDOW_NORMAL)
        cv2.resizeWindow(win_name, new_w, new_h)
        cv2.moveWindow(win_name, cur_x, 0)
        cur_x += new_w
        resized_image = cv2.resize(images_arr[i], (new_w, new_h))
        cv2.imshow(win_name, (resized_image * 255).astype(np.uint8))

    cv2.waitKey(0)
    cv2.destroyAllWindows()

# неблокирующий просмотр изображений(np массивов)
def show_images_parallel(images_arr):
    thread = threading.Thread(target = show_images, args = [images_arr])
    thread.start()
    return thread

In [4]:
pcd = o3d.io.read_point_cloud("test_pcd.ply")
# pcd_obj = o3d.data.PLYPointCloud()
# pcd = o3d.io.read_point_cloud(pcd_obj.path)

In [5]:
print(f"Исходное количество точек: {len(pcd.points)}")

start = time.time()
cur_time = start

downsampled_pcd = downsample(pcd)

timer("Даунсемплинг(случайный)")

#Ищем плоскость пола и точки в ней и вне неё
plane_model, _ = downsampled_pcd.segment_plane(distance_threshold = distance_threshold,
                                               num_iterations = ransac_iterations,
                                               ransac_n = 3)
[a, b, c, d] = plane_model
#строим нормаль к поверхности   
normal = np.array([a, b, c])

timer("Поиск плоскости пола")

# Убираем слишком далекие от пола точки
pcd_points = np.asarray(pcd.points)
close_mask = np.abs(a * pcd_points[:, 0] 
                  + b * pcd_points[:, 1] 
                  + c * pcd_points[:, 2] 
                  + d) <= max_dist
close_points = pcd_points[close_mask]

timer("Выбор близких к полу точек")

close_points_ds = voxel_downsample_fast(close_points, grid_step)

timer("Voxel-Даунсемплинг близких точек")

# Разделяем на inliers и outliers
close_distances_ds = np.abs(np.dot(close_points_ds, normal) + d)
inlier_mask = close_distances_ds <= distance_threshold
inliers = close_points_ds[inlier_mask]
outliers = close_points_ds[~inlier_mask]

timer("Выделение точек в и вне плоскости")

#меняем направление нормали в сторону большего количества точек
condition = np.dot(outliers, normal) + d < 0
if len(outliers[condition]) > len(outliers) // 2:
    normal = -normal

# берем 2 перпендикулярных вектора в плоскости
v1_norm = np.linalg.norm([b, -a, 0])
plane_v1 = np.asarray([b, -a, 0]) / v1_norm
plane_v2 = np.linalg.cross(normal, plane_v1)

#проецируем точки пола на плоскость пола
projected = project_to_plane(inliers, plane_model)

timer("Проекция точек пола на плоскость")

#переходим от координат в пространстве в координаты 2-х векторов в плоскости
plane_coords, x_max, x_min, y_max, y_min = switch_to_plane_coords(projected, plane_v1, plane_v2)

#строим двумерный массив, соответствующий плоскости
floor = np.zeros((
    int((y_max - y_min) // grid_step + 1), 
    int((x_max - x_min) // grid_step + 1)
))
# Инвертируем Y т.к. в opencv 0 по оси Y это самый верх
x_indices = ((plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((y_max - plane_coords[:, 1]) // grid_step).astype(int) 
valid_mask = (x_indices >= 0) & (x_indices < floor.shape[1]) & \
             (y_indices >= 0) & (y_indices < floor.shape[0])

floor[y_indices[valid_mask], x_indices[valid_mask]] = 1

# наращиваем точки пола
kernel = np.ones((morph_fill_size,morph_fill_size))
floor = cv2.morphologyEx(floor, cv2.MORPH_CLOSE, kernel)
floor_grid = np.copy(floor)

timer("Составление сетки пола")

# проецируем точки вне пола на плоскость
projected_outliers = project_to_plane(outliers, plane_model)

timer("Проекция точек препятствия на пол")

# убираем из плоскости точки, где есть препятствия
outliers_plane_coords = switch_to_plane_coords(projected_outliers, plane_v1, plane_v2)[0]
x_indices = ((outliers_plane_coords[:, 0] - x_min) // grid_step).astype(int)
y_indices = ((y_max - outliers_plane_coords[:, 1]) // grid_step).astype(int)
valid_mask = (x_indices >= 0) & (x_indices < floor.shape[1]) & \
             (y_indices >= 0) & (y_indices < floor.shape[0])

floor_obstacles = floor
floor_obstacles[y_indices[valid_mask], x_indices[valid_mask]] = 0

# наращиваем точки пола
kernel = np.ones((morph_size,morph_size))
floor_obstacles = cv2.morphologyEx(floor_obstacles, cv2.MORPH_OPEN, kernel)

timer("Составление сетки пола с препятствиями")
print(f"Общее время: {time.time() - start:.4f} секунд")

Исходное количество точек: 13002443
Даунсемплинг(случайный): 0.8206 секунд
Поиск плоскости пола: 0.1078 секунд
Выбор близких к полу точек: 0.3695 секунд
Voxel-Даунсемплинг близких точек: 1.3149 секунд
Выделение точек в и вне плоскости: 0.0625 секунд
Проекция точек пола на плоскость: 0.0295 секунд
Составление сетки пола: 0.0907 секунд
Проекция точек препятствия на пол: 0.0307 секунд
Составление сетки пола с препятствиями: 0.0883 секунд
Общее время: 2.9146 секунд


In [11]:
floor_accessible = np.copy(floor_obstacles)

#убираем маленькие компоненты 
components_info = cv2.connectedComponentsWithStats(
    floor_accessible.astype(np.uint8), 
    connectivity=4, 
    ltype=cv2.CV_32S
)
[num_labels, labels, stats, centroids] = components_info
for i in range(1, num_labels):
    area = stats[i, cv2.CC_STAT_AREA]*grid_step**2
    if area < min_area:
        floor_accessible[labels == i] = 0

# сначала жестко так прям убираем шум
kernel = np.ones((morph_fill_size, morph_fill_size))
floor_polished = cv2.morphologyEx(floor_accessible, cv2.MORPH_CLOSE, kernel)

floor_expanded = np.copy(floor_polished)

# теперь возвращаем препятствия, которые стерлись как шум
floor_accessible_reversed = 1 - floor_accessible
floor_polished_reversed = 1 - floor_expanded
components_info = cv2.connectedComponentsWithStats(
    floor_accessible_reversed.astype(np.uint8), 
    connectivity=4, 
    ltype=cv2.CV_32S
)
[num_labels, labels, stats, centroids] = components_info
for i in range(1, num_labels):
    obstacle = floor_accessible_reversed[labels == i]
    if np.any(cv2.bitwise_and(obstacle, floor_polished_reversed[labels == i])):
        floor_expanded[labels == i] = 0

pcd_thread = show_pcd_concurrent(downsampled_pcd)
im_thread = show_images_parallel([floor_obstacles, floor_accessible, floor_expanded])

im_thread.join()
pcd_thread.join()

In [7]:
# unused visualization

# inliers_pcd = o3d.geometry.PointCloud()
# inliers_pcd.points = o3d.utility.Vector3dVector(inliers)
# inliers_pcd.paint_uniform_color([0, 1, 0])

# outliers_pcd = o3d.geometry.PointCloud()
# outliers_pcd.points = o3d.utility.Vector3dVector(outliers)
# outliers_pcd.paint_uniform_color([1, 0, 0])

# visualise_concurrent(inliers_pcd)
# visualise_concurrent(outliers_pcd)
# visualise(projected)
# visualise(projected_outliers)

# # Точки близкие к полу
# visualise(close_points)

# #выделенная плоскость
# inliers_pcd = o3d.geometry.PointCloud()
# inliers_pcd.points = o3d.utility.Vector3dVector(inliers)

# outliers_pcd = o3d.geometry.PointCloud()
# outliers_pcd.points = o3d.utility.Vector3dVector(outliers)

# inliers_pcd.paint_uniform_color([0, 1, 0])
# outliers_pcd.paint_uniform_color([1, 0, 0])
# vis.draw_geometries([inliers_pcd, outliers_pcd])

# #проекция точек пола на плоскость пола
# visualise(projected)

# # Проекция точек вне пола на плоскость пола
# visualise(projected_outliers)
